### csv 피처들만 남기기

In [56]:
import pandas as pd

for i in range(17,26):
    # 인코딩이 euc-kr로 되어있음
    telecom_df = pd.read_csv(f'../../data/raw/p{i}v32_KMP_csv.csv', encoding='euc-kr')
    telecom_df = telecom_df.rename(columns={
        f'p{i}pid': 'id',
        f'p{i}gender': 'gender',
        f'p{i}byear': 'birth_year',
        f'p{i}school': 'school',
        f'p{i}mar':'mar',
        f'p{i}income1':'income',
        f'p{i}job1':'job',
        f'p{i}area':'region',
        f'p{i}area':'region',
        f'p{i}c02003':'mobile_bundle',
        f'p{i}c01001':'phone_usage_per_m',
        f'p{i}a03008':'telecom'
    })
    telecom_df['year'] = f'20{i}'
    # 나이 컬럼 추가
    telecom_df['age'] = telecom_df['year'].astype(int) - telecom_df['birth_year']
    telecom_df = telecom_df[['id', 'gender', 'age', 'year', 'school', 'mar', 'mobile_bundle', 'income', 'job', 'region', 'phone_usage_per_m', 'telecom']]
    
    # 저장할 때 utf-8로 변환
    telecom_df.to_csv(f"../../data/processed/p{i}v32_KMP_csv.csv", index=False, encoding="utf-8")

C:\Users\Playdata\AppData\Local\Temp\ipykernel_27064\3005749850.py:5: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  telecom_df = pd.read_csv(f'../../data/raw/p{i}v32_KMP_csv.csv', encoding='euc-kr')
C:\Users\Playdata\AppData\Local\Temp\ipykernel_27064\3005749850.py:5: DtypeWarning: Columns (4,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  telecom_df = pd.read_csv(f'../../data/raw/p{i}v32_KMP_csv.csv', encoding='euc-kr')
C:\Users\Playdata\AppData\Local\Temp\ipykernel_27064\3005749850.py:5: DtypeWarning: Columns (5,8,9,73) have mixed types. Specify dtype option on import or set low_memory=False.
  telecom_df = pd.read_csv(f'../../data/raw/p{i}v32_KMP_csv.csv', encoding='euc-kr')
C:\Users\Playdata\AppData\Local\Temp\ipykernel_27064\3005749850.py:5: DtypeWarning: Columns (5,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  telecom_df = pd.read_csv(f'../../data/raw/p{i}v3

### csv 합치기

In [57]:
import glob

files = glob.glob("../../data/processed/*.csv")

df_list = [pd.read_csv(f) for f in files]
merged_df = pd.concat(df_list, ignore_index=True)

merged_df.to_csv("../../data/processed/merged_telecom.csv", index=False)

### csv 결측치 제거

In [58]:
telecom_df = pd.read_csv('../../data/processed/merged_telecom.csv')
telecom_df = telecom_df[(telecom_df != " ").all(axis=1)]
telecom_df_count = telecom_df.groupby('id')['id'].transform('count')
telecom_df.to_csv("../../data/processed/merged_telecom.csv", index=False)

In [59]:
# 같은 pid값에 9개씩 있는지 확인
pd.set_option('display.max_rows', None)
telecom_df['id'].value_counts()
A = telecom_df['id'].value_counts()
A.value_counts()

count
9    3819
7    2441
2    1254
8    1027
1     939
6     907
5     894
3     663
4     644
Name: count, dtype: int64

### 통신사 이탈 여부 컬럼 추가
- 작년과 비교했을 때 통신사가 달라졌으면 이탈로 간주
- 이탈했을 경우 1 아닌 경우 0
- 2017은 비교 대상이 없으니 0

In [60]:
telecom_df = pd.read_csv('../../data/processed/merged_telecom.csv')
pd.set_option('display.max_rows', None)

# 아이디,년도 별로 정렬
telecom_df = telecom_df.sort_values(['id', 'year'])

# 아이디별로 묶어 만약 이전 row와 값이 1
telecom_df['telecom_change'] = telecom_df.groupby('id')['telecom'].shift(1)
telecom_df['telecom_change_yn'] = (telecom_df['telecom_change'] != telecom_df['telecom']).astype(int)

telecom_df['telecom_change'] = telecom_df['telecom_change'].astype('Int64')
telecom_df = telecom_df.drop('telecom_change', axis=1)
# 2017은 이전 값이 없으니 0
telecom_df.loc[telecom_df['year']==2015, 'telecom_change_yn'] = 0
#telecom_df.loc[telecom_df['year']==2017, 'telecom_change'] = telecom_df['telecom']
telecom_df.to_csv("../../data/processed/telecom.csv", index=False)

### job, gender 컬럼 0, 1로 변환

In [61]:
telecom_df = pd.read_csv('../../data/processed/telecom.csv')
telecom_df['gender'] = telecom_df['gender'].replace({1: 0, 2: 1})
telecom_df['job'] = telecom_df['job'].replace({1: 1, 2: 0})
telecom_df['mobile_bundle'] = telecom_df['mobile_bundle'].replace({1: 1, 2: 0})
telecom_df.to_csv("../../data/processed/telecom.csv", index=False)

### 인코딩

In [62]:
import pandas as pd
telecom_df = pd.read_csv('../../data/processed/telecom.csv')
telecom_df['school'] = telecom_df['school']-1
telecom_df['income'] = telecom_df['income']-1
df_encoded = pd.get_dummies(telecom_df, columns=['mar', 'region', 'telecom'], dtype='int')

df_encoded.to_csv("../../data/processed/telecom_encoding.csv", index=False)

In [63]:
telecom_df = pd.read_csv("../../data/processed/telecom_encoding.csv")
telecom_df = telecom_df.rename(columns={
    'region_1': 'seoul',
    'region_2': 'busan',
    'region_3': 'daegu',
    'region_4': 'incheon',
    'region_5': 'gwangju',
    'region_6': 'daejeon',
    'region_7': 'ulsan',
    'region_8': 'gyeonggi',
    'region_9': 'gangwon',
    'region_10': 'chungbuk',
    'region_11': 'chungnam',
    'region_12': 'jeonbuk',
    'region_13': 'jeonnam',
    'region_14': 'gyeongbuk',
    'region_15': 'gyeongnam',
    'region_16': 'jeju',
    'region_17': 'sejong',
    'telecom_1': 'skt',
    'telecom_2': 'kt',
    'telecom_3': 'lgu',
    'telecom_4': 'mvno',
})
telecom_df = telecom_df[telecom_df['year']!=2015]
telecom_df = telecom_df[telecom_df['year']!=2016]
telecom_df = telecom_df[telecom_df['year']!=2017]
# 저장할 때 utf-8로 변환
telecom_df.to_csv("../../data/processed/telecom_encoding.csv", index=False)

In [64]:
telecom_df = pd.read_csv("../../data/processed/telecom_encoding.csv")
telecom_df['telecom_change_yn'].value_counts()

telecom_change_yn
0    40642
1    29503
Name: count, dtype: int64